In [1]:
# Select GPU for Execution
# 1. Click on Runtime > change Runtime type
# 2. Select T4 GPU

In [2]:
# Create a model that can predict the salary of teh employee based on employee's years of experience

In [3]:
!nvidia-smi

Sun May 31 06:04:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import pandas as pd
import numpy as np

In [5]:
data = pd.read_csv("Salary_Data.csv")

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   YearsExperience  30 non-null     float64
 1   Salary           30 non-null     float64
dtypes: float64(2)
memory usage: 628.0 bytes


In [7]:
data.dropna(inplace=True)

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30 entries, 0 to 29
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   YearsExperience  30 non-null     float64
 1   Salary           30 non-null     float64
dtypes: float64(2)
memory usage: 720.0 bytes


In [9]:
# Rules for Regression using ANN
# ============================================================================
# 1. Data must be COMPLETE
# 2. Dtaa must be STRICTLY NUMERIC
# 3. Features and Label must be represented in the form 2d numpy array
# 4. Normalizing/Standardizing Features and Label is MANDATORY.

#Regarding 4th point, guidelines by PN
#=============================================================================
# 1. Features must be Standardized ,
#             if feature cols are NORMALLY DISTRIBUTED:
#                        use STANDARDSCALER
#             else:
#                        use RobustScaler
# 2. Label must be standardized using MINMAXSCALER with range 0 to 1

In [10]:
# Seperate data as features and label

features = data.iloc[:,[0]].values
label = data.iloc[:,[1]].values

In [11]:
#Feature Scaling

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
features = sc.fit_transform(features)


In [12]:
# Label Scaling

from sklearn.preprocessing import MinMaxScaler
ms = MinMaxScaler()
label = ms.fit_transform(label)

In [13]:
# Create train test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(features,
                                                 label,
                                                 test_size=0.2,
                                                 random_state=1)

# Modelling Phase Starts

1. Architect the model
2. Compile model
3. Fit training data to the model
4. Check the quality of the model
5. Deploy model

In [14]:
import tensorflow as tf

In [15]:
tf.__version__

'2.20.0'

In [16]:
#Step1: Architect model

model = tf.keras.Sequential()

#InputLayer + h1
model.add(tf.keras.layers.Dense(units = 100, activation='sigmoid', input_shape=(1,)))

#h2
model.add(tf.keras.layers.Dense(units = 100, activation='sigmoid'))

#h3
model.add(tf.keras.layers.Dense( units= 100, activation='sigmoid'))

#output layer
model.add(tf.keras.layers.Dense( units = 1, activation='linear'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,501 (80.08 KB)

 Trainable params: 20,501 (80.08 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Step2: Compile Model

# The goal of this step is to declare the following:
# 1. Which Back propogation algo to use? --------------> SGD (Stochastic Gradient Descent)
# 2. Which error function to use? ---------------------> MSE
# 3. Which evaluation metrics to use? -----------------> R2 score


model.compile(optimizer="sgd", #declaring with back prop algo
              loss="mean_squared_error", #declaring error function
              metrics=[tf.keras.metrics.R2Score()]) #declaring metric function for quality check

In [19]:
# Step3: Train the model

model.fit(X_train,y_train,
          validation_data=(X_test,y_test),
          epochs=6000)

Streaming output truncated to the last 5000 lines.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - loss: 0.0317 - r2_score: 0.7308 - val_loss: 0.0175 - val_r2_score: 0.4182
Epoch 3502/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.0316 - r2_score: 0.7312 - val_loss: 0.0175 - val_r2_score: 0.4186
Epoch 3503/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - loss: 0.0316 - r2_score: 0.7315 - val_loss: 0.0175 - val_r2_score: 0.4190
Epoch 3504/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - loss: 0.0315 - r2_score: 0.7319 - val_loss: 0.0174 - val_r2_score: 0.4194
Epoch 3505/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 0.0315 - r2_score: 0.7323 - val_loss: 0.0174 - val_r2_score: 0.4198
Epoch 3506/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - loss: 0.0315 - r2_score: 0.7327 - val_loss: 0.0174 - val_r2_score: 0.4202
Epoch 3507/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 0.0314 - r2_score: 0.7331 - val_loss: 0.0174 - val_r2_score: 0.4206
Epoch 3508/6000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150m